# `spectrometry` tutorial

This notebook teaches the UV-Vis reader and sample-aggregation tools in `empylib.spectrometry`. The examples use the packaged sample files in `docs/` so you can reproduce the workflow offline.

**Learning goals**

- read a single UV-Vis export file into a tidy DataFrame
- scan a folder to discover available sample names automatically
- aggregate all measurements belonging to one sample into a combined reflectance / transmittance table

**Notebook design**

- every runnable cell calls the public `empylib` API directly
- parameter meanings are explained in markdown and in short inline comments
- outputs are inspected in the same notebook so you can see what each function returns
- the core path is offline-first; internet-backed examples live in clearly marked optional appendices

In [ ]:
from pathlib import Path
import os
import sys

current = Path.cwd().resolve()
for candidate in (current, *current.parents):
    if (candidate / "empylib").exists() and (candidate / "docs").exists():
        ROOT = candidate
        break
else:
    raise FileNotFoundError("Could not locate the EMPI Lib repository root.")

if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from IPython.display import display
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

plt.rcParams["figure.figsize"] = (7, 3)

import empylib.spectrometry as spec

DOCS = ROOT / "docs"
UVVIS_DIR = DOCS / "uvvis_sample_data"
PERKIN_PATH = DOCS / "PerkinElmer_sample.asc"

IndentationError: unexpected indent (3370489915.py, line 2)

## Read one UV-Vis file

**Functions used**

- spec.read_uvvis

**Problem we are solving**

The basic workflow starts with a single exported file. `read_uvvis` detects the vendor format, parses the data, converts wavelengths to micrometers, and scales percent columns to fractions when needed.

**Parameter guide for this example**

- `path`: path to the UV-Vis export file
- `vendor`: optional manual override if the extension is ambiguous
- `col1_name` and `col2_name`: optional manual column names if the file headers are incomplete

**Outputs to inspect**

- a DataFrame indexed by wavelength in micrometers
- metadata in `df.attrs`, such as instrument and sample name

In [ ]:
uvvis_single = spec.read_uvvis(
    str(PERKIN_PATH),  # packaged PerkinElmer example file
)

display(uvvis_single.head())
print("Instrument:", uvvis_single.attrs.get("instrument"))
print("Sample name:", uvvis_single.attrs.get("sample_name"))

**How to read the result**

The index is already in micrometers and the measurement column is scaled to a 0-1 fraction when the raw file looked like percent data. This is the right format for the rest of the library.

**Common pitfalls**

- If the instrument headers are incomplete, provide `col1_name` or `col2_name` manually
- The function infers the vendor from the file extension when `vendor` is omitted

**Try this next**

- Read one of the sample files inside `UVVIS_DIR` and compare the metadata
- Override `col2_name` to make the output column label match your preferred naming convention

## Find the samples available in a folder

**Functions used**

- spec.find_uvvis_samples

**Problem we are solving**

Large UV-Vis campaigns often generate several files per sample. `find_uvvis_samples` scans the naming patterns and returns the unique sample names it can infer.

**Parameter guide for this example**

- `search_dirs`: folders scanned for UV-Vis files
- `tags`: optional measurement tags such as `Rtot`, `Ttot`, `Rspec`, and `Tspec`
- `aliases`: optional alternate prefixes if your instrument naming differs from the defaults

**Outputs to inspect**

- `sample_names`: list of discovered sample identifiers

In [ ]:
sample_names = spec.find_uvvis_samples(
    search_dirs=[str(UVVIS_DIR)],
)

print("Discovered samples:")
for name in sample_names:
    print("-", name)

**How to read the result**

The returned names are the sample identifiers inferred from the filenames after removing the measurement tag prefix. This is useful when you want a notebook or script to discover data automatically instead of hardcoding sample names.

**Common pitfalls**

- This helper depends on filename patterns, so inconsistent naming conventions can hide valid files
- If your tags differ from the defaults, pass custom `tags` and `aliases`

**Try this next**

- Restrict the scan to a smaller tag list if you only care about reflectance or only about transmittance
- Use the discovered sample names in a loop with `sample_uvvis`

## Aggregate all files for one sample

**Functions used**

- spec.sample_uvvis

**Problem we are solving**

A full sample usually includes several measurements: total, specular, and diffuse reflectance or transmittance. `sample_uvvis` reads all matching files, interpolates them onto a common grid, and combines them into one DataFrame.

**Parameter guide for this example**

- `sample`: sample name as it appears in the filenames after the measurement tag
- `search_dirs`: folders where the measurement files live
- `ref_standard='spectralon'`: reflectance standard used for the correction of reflectance channels

**Outputs to inspect**

- a DataFrame with columns such as `Rtot`, `Rspec`, `Rdif`, `Ttot`, `Tspec`, and `Tdif` when available
- sample metadata stored in `dataset.attrs`

In [ ]:
uvvis_sample = spec.sample_uvvis(
    sample="PMMApaint_CaCO3_15vv",
    search_dirs=[str(UVVIS_DIR)],
)

display(uvvis_sample.head())
print("Columns:", list(uvvis_sample.columns))
print("Sample attribute:", uvvis_sample.attrs.get("sample_name"))

fig, ax = plt.subplots()
for col in uvvis_sample.columns:
    ax.plot(uvvis_sample.index, uvvis_sample[col], label=col)
ax.set_xlabel("Wavelength (um)")
ax.set_ylabel("Measured fraction")
ax.set_title("Aggregated UV-Vis sample")
ax.grid(True, alpha=0.3)
ax.legend()
plt.show()

**How to read the result**

This DataFrame is the practical endpoint of a UV-Vis import workflow: one wavelength index, several corrected measurement channels, and consistent naming that is easy to feed into later optical calculations.

**Common pitfalls**

- If one expected file is missing, the returned DataFrame may still be valid but have fewer columns
- The function interpolates onto the union grid of the discovered files, so small NaN patterns can appear if a measurement does not cover the full range

**Try this next**

- Loop over all discovered sample names and store the resulting DataFrames in a dictionary
- Feed one reflectance spectrum into `color_system.spectrum_to_hex` to estimate the visible color